# **Testing and Distribution**

**DS212 — Object-Oriented Programming**

**Week 14:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Test a Python package locally and prepare a controlled distribution workflow using GitHub and TestPyPI.

A package is only useful if people can **trust** it and **install** it. This lesson covers both halves:

```{mermaid}
flowchart LR
    W["Write code"] --> T["Unit tests<br/>verify behavior"]
    T --> L["Local install<br/>pip install -e ."]
    L --> B["Build<br/>sdist + wheel"]
    B --> G["GitHub<br/>push, tag, CI"]
    G --> P["TestPyPI<br/>upload & install"]
```

## **Unit Testing and Verification**

A **unit test** checks one small piece of behavior — usually one method — in isolation. Tests turn the question *"does my code still work?"* into a command you can run in seconds after every change.

### **Anatomy of a test: Arrange – Act – Assert**

```python
def test_average_of_recorded_scores():
    student = Student("001", "Ana")      # Arrange: set up objects
    student.add_score(90)
    student.add_score(80)

    result = student.average()           # Act: call the behavior

    assert result == 85                  # Assert: check the outcome
```

### **`unittest` (standard library)**

Python includes the `unittest` framework. Tests are **methods of a class** that inherits from `unittest.TestCase` — an object-oriented design in itself.

In [ ]:
import sys
import unittest
from pathlib import Path

project = Path("..") / "files" / "gradebook-project"
sys.path.insert(0, str(project / "src"))

from gradebook import Gradebook, InvalidScoreError, Student, StudentNotFoundError


class TestStudent(unittest.TestCase):

    def setUp(self):                         # runs before EACH test: a fresh object
        self.student = Student("2026-001", "Ana")

    def test_average(self):
        for score in (90, 80, 85):
            self.student.add_score(score)
        self.assertAlmostEqual(self.student.average(), 85.0)

    def test_average_without_scores(self):
        self.assertEqual(self.student.average(), 0.0)

    def test_invalid_score_raises(self):     # testing that errors are raised
        with self.assertRaises(InvalidScoreError):
            self.student.add_score(150)

    def test_boundaries_are_accepted(self):  # edge cases: exactly 0 and 100
        self.student.add_score(0)
        self.student.add_score(100)
        self.assertEqual(self.student.scores, (0, 100))


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestStudent)
unittest.TextTestRunner(verbosity=2, stream=sys.stdout).run(suite);

Common assertion methods:

| Method                          | Checks                         |
| ------------------------------- | ------------------------------ |
| `assertEqual(a, b)`             | `a == b`                       |
| `assertAlmostEqual(a, b)`       | floats are equal after rounding |
| `assertTrue(x)` / `assertFalse(x)` | truthiness                  |
| `assertIn(item, container)`     | membership                     |
| `assertIsInstance(obj, cls)`    | type                           |
| `assertRaises(Error)`           | the code raises `Error`        |

### **What should you test?**

* the **normal case** (typical inputs);
* **edge cases** (empty lists, zero, exact boundaries such as 0 and 100);
* **error cases** (invalid input raises the right exception);
* **interactions** between objects (adding a student to a gradebook updates its length).

A failing test is useful — it tells you exactly what broke:

In [ ]:
class TestBuggyExpectation(unittest.TestCase):
    def test_wrong_expectation(self):
        book = Gradebook("DS212")
        book.add(Student("001", "Ana"))
        book.record("001", 70)
        self.assertEqual(book.average("001"), 75)   # deliberately wrong


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestBuggyExpectation)
result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
print("failures:", len(result.failures))

### **Running a project's whole test suite**

In a real project, tests live in `tests/` and run from the command line. The sample project's tests can be discovered and run like this:

In [ ]:
suite = unittest.defaultTestLoader.discover(
    start_dir=str(project / "tests"), top_level_dir=str(project)
)
result = unittest.TextTestRunner(verbosity=1, stream=sys.stdout).run(suite)
print("all passed:", result.wasSuccessful())

From a terminal in the project root, the equivalent commands are:

```bash
python -m unittest discover -s tests      # standard library
python -m pytest                          # pytest (also runs unittest tests)
```

### **pytest**

Most Python projects use **pytest**, a third-party framework (installed with `pip install pytest`) that needs less boilerplate: tests are plain functions using plain `assert`.

```python
# tests/test_gradebook_pytest.py
import pytest
from gradebook import Gradebook, Student, StudentNotFoundError


@pytest.fixture
def book():                                   # reusable setup
    b = Gradebook("DS212")
    b.add(Student("001", "Ana"))
    return b


def test_record_and_average(book):
    book.record("001", 90)
    assert book.average("001") == 90


def test_unknown_student(book):
    with pytest.raises(StudentNotFoundError):
        book.record("999", 80)


@pytest.mark.parametrize("score", [-1, 101, 250])
def test_invalid_scores(book, score):         # one test, many inputs
    with pytest.raises(Exception):
        book.record("001", score)
```

```text
$ python -m pytest -q
.....                                                     [100%]
5 passed in 0.03s
```

## **Local Package Installation and Testing**

Before sharing a package, install it **locally** exactly as a user would.

### **Editable install for development**

```bash
cd gradebook-project
python -m venv .venv
.venv\Scripts\activate
pip install -e ".[dev]"
```

`-e` (editable) links the installed package to your `src/` folder, so code changes take effect immediately without reinstalling.

### **Building distributions**

```bash
python -m build
```

```text
dist/
├── ds212_gradebook_yourname-0.1.0.tar.gz            ← source distribution (sdist)
└── ds212_gradebook_yourname-0.1.0-py3-none-any.whl  ← wheel (ready to install)
```

### **Testing the built wheel in a clean environment**

The most reliable check is a **fresh** virtual environment, so nothing from your development folder leaks in:

```bash
python -m venv /tmp/check-env
/tmp/check-env/Scripts/activate
pip install dist/ds212_gradebook_yourname-0.1.0-py3-none-any.whl
python -c "import gradebook; print(gradebook.__version__)"
```

If this works, users will be able to install your package too.

## **GitHub Workflow**

GitHub hosts the repository, its history, and its **releases**.

```bash
git remote add origin https://github.com/your-username/ds212-gradebook.git
git push -u origin main
git tag -a v0.1.0 -m "First release"
git push origin v0.1.0
```

On GitHub you can then create a **Release** from the tag, attaching the files in `dist/` and the changelog notes.

### **Continuous integration (CI)**

**GitHub Actions** can run your tests automatically on every push, so a broken change is detected immediately. A minimal workflow file:

```yaml
# .github/workflows/tests.yml
name: tests
on: [push, pull_request]
jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install -e ".[dev]"
      - run: python -m pytest
```

## **TestPyPI Workflow**

**PyPI** (pypi.org) is the public index that `pip install` uses. **TestPyPI** (test.pypi.org) is a **separate practice index** where you can rehearse publishing without affecting real users.

```{mermaid}
sequenceDiagram
    participant Dev as Developer
    participant TP as TestPyPI
    participant U as Clean environment
    Dev->>Dev: python -m build
    Dev->>Dev: twine check dist/*
    Dev->>TP: twine upload --repository testpypi dist/*
    U->>TP: pip install --index-url https://test.pypi.org/simple/ ds212-gradebook-yourname
    TP-->>U: wheel
    U->>U: import gradebook
```

### **Steps**

1. Create an account at **test.pypi.org** and generate an **API token** (Account settings → API tokens).
2. Make sure the `name` in `pyproject.toml` is **unique** (e.g., add your username).
3. Build and check the distributions:

   ```bash
   python -m build
   python -m twine check dist/*
   ```

4. Upload. When prompted, use `__token__` as the username and paste your token as the password:

   ```bash
   python -m twine upload --repository testpypi dist/*
   ```

5. Install it from TestPyPI in a clean environment:

   ```bash
   pip install --index-url https://test.pypi.org/simple/ --no-deps ds212-gradebook-yourname
   ```

> **Security:** an API token is a password. **Never** paste it into a notebook, commit it to Git, or share it in a group chat. If a token leaks, revoke it immediately on the website.

> **Versions are permanent.** A version number can be uploaded only **once**. To fix a mistake, bump the version (`0.1.0 → 0.1.1`) and upload again.

## **Distribution Checklist**

* [ ] tests pass locally (`python -m pytest`) and in CI;
* [ ] `pip install -e .` works in a fresh virtual environment;
* [ ] `python -m build` produces an sdist and a wheel;
* [ ] `twine check dist/*` reports no problems;
* [ ] the wheel installs and imports in a **clean** environment;
* [ ] the version is bumped, the changelog updated, and the release tagged;
* [ ] the package installs from TestPyPI.

## **Summary**

| Concept             | Key idea                                                              |
| ------------------- | --------------------------------------------------------------------- |
| unit test           | checks one behavior: arrange, act, assert                             |
| `unittest`          | standard-library, class-based tests (`TestCase`, `setUp`, `assert…`)  |
| pytest              | function-based tests with plain `assert`, fixtures, parametrization   |
| editable install    | `pip install -e .` — develop and test the installed package           |
| build               | `python -m build` creates an sdist and a wheel                        |
| CI                  | GitHub Actions runs tests on every push                               |
| TestPyPI            | practice index for rehearsing a release with `twine`                  |

## **References**

Beck, K. (2002). *Test-driven development: By example*. Addison-Wesley.

Chacon, S., & Straub, B. (2014). *Pro Git* (2nd ed.). Apress. https://git-scm.com/book/en/v2

GitHub. (2026). *GitHub docs: Releases*. https://docs.github.com/en/repositories/releasing-projects-on-github/about-releases

pytest-dev. (2026). *pytest documentation*. https://docs.pytest.org/

Python Packaging Authority. (2026). *Packaging Python projects*. https://packaging.python.org/en/latest/tutorials/packaging-projects/

Python Packaging Authority. (2026). *Python packaging user guide*. https://packaging.python.org/

Python Packaging Authority. (2026). *Using TestPyPI*. https://packaging.python.org/en/latest/guides/using-testpypi/

Python Software Foundation. (2026). *unittest — Unit testing framework*. https://docs.python.org/3/library/unittest.html